# 用 AI × Colab（R）分析 Titanic 乘客資料

大數據分析與實踐 · R 寶的 AI × Colab 實戰。這份筆記本使用 **R 執行階段**（如果打開後不是 R：執行階段 → 變更執行階段類型 → R）。每一步上面的引用框是當時給 AI 的 prompt，下面是 AI 寫好、我們檢查過的程式碼。

教材網頁：https://chenyc0901.github.io/R_datastructure_learning/colab.html

**🤖 給 AI 的 prompt**

> 你是一位資料分析助教，熟悉 R 的 tidyverse（readr、dplyr、tidyr、ggplot2）。
> 我是醫技系的大學生，正在 Google Colab 的 R 執行階段分析 Titanic 乘客資料（891 位乘客，欄位有 PassengerId、Survived、Pclass、Name、Sex、Age、SibSp、Parch、Ticket、Fare、Cabin、Embarked）。
> 接下來我會一步一步請你寫程式，請遵守：
> 1. 一律用 R，使用 tidyverse 的寫法和 |> 管線，不要給我 Python。
> 2. 每次只做我要求的那一步，不要一次做完全部。
> 3. 程式碼加上繁體中文註解。
> 4. 圖上的文字一律用英文（Colab 預設沒有中文字型）。
> 5. 不要自己編造數字；結果以我執行後貼回給你的輸出為準。

## 步驟 1　讀取資料

**🤖 給 AI 的 prompt**

> 請寫 R 程式：載入 tidyverse，從下面的網址讀取 CSV，存到變數 df，印出它有幾列幾欄，再顯示前幾列。
> 另外請設定 Colab 裡圖的大小為寬 8、高 5。
> https://chenyc0901.github.io/R_datastructure_learning/data/titanic.csv

In [ ]:
library(tidyverse)                                   # 載入 readr、dplyr、tidyr、ggplot2……
options(repr.plot.width = 8, repr.plot.height = 5)   # Colab 裡圖的寬、高（英吋）

url <- "https://chenyc0901.github.io/R_datastructure_learning/data/titanic.csv"
df <- read_csv(url, show_col_types = FALSE)          # 讀 CSV，存成 tibble
dim(df)                                              # 列數、欄數
head(df)                                             # 前 6 列

## 步驟 2　認識資料

**🤖 給 AI 的 prompt**

> 請用 glimpse() 和 summary() 檢查 df：每一欄的資料型態、缺值，以及數值欄位的摘要統計。
> 執行結果我貼在下面，請用白話說明每一欄代表什麼，以及你看到哪些需要注意的地方。
> （貼上輸出）

In [ ]:
glimpse(df)                    # 每一欄的型態和前幾個值

In [ ]:
summary(df)                    # 數值欄：最小、四分位數、平均、最大、NA 數

## 步驟 3　缺值與清理

**🤖 給 AI 的 prompt**

> 請算出 df 每一欄有幾個缺值、佔全部的百分比，整理成一張表（一欄一列），由多到少排列。

In [ ]:
df |>
  summarise(across(everything(), \(x) sum(is.na(x)))) |>               # 每一欄數缺值
  pivot_longer(everything(), names_to = "column", values_to = "n_missing") |>
  mutate(percent = round(n_missing / nrow(df) * 100, 1)) |>              # 佔全部的 %
  arrange(desc(n_missing))

**🤖 給 AI 的 prompt**

> Cabin 缺 687 筆、Age 缺 177 筆、Embarked 缺 2 筆。
> 請先不要寫程式，列出每一欄可以怎麼處理（刪掉欄位、補值、保留缺值），以及各自的優缺點。

**🤖 給 AI 的 prompt**

> 好，請照下面的決定寫程式，結果存成新的 tibble clean，不要改動 df：
> 1. 刪掉 Cabin 和 Ticket 兩欄。
> 2. Embarked 用出現最多次的值補。
> 3. Age 保留缺值，不要補。
> 4. Pclass 轉成 factor（它是類別，不是數字）。
> 5. 新增 FamilySize = SibSp + Parch + 1。
> 6. 新增 AgeGroup，用 cut() 把 Age 分成 0-12、13-18、19-40、41-60、61-80。
> 7. 新增 Outcome，把 Survived 的 0 / 1 換成 "Died" / "Survived"。
> 最後檢查 clean 每一欄還有幾個缺值。

In [ ]:
most <- names(which.max(table(df$Embarked)))      # 出現最多次的港口
most

clean <- df |>
  select(-Cabin, -Ticket) |>                        # 刪掉兩欄（df 本身不會變）
  mutate(Embarked   = replace_na(Embarked, most),   # 只缺 2 筆，補最常見的港口
         Pclass     = factor(Pclass),               # 艙等是類別
         FamilySize = SibSp + Parch + 1,            # 兄弟姊妹配偶 + 父母子女 + 自己
         AgeGroup   = cut(Age, breaks = c(0, 12, 18, 40, 60, 80),
                          labels = c("0-12", "13-18", "19-40", "41-60", "61-80")),
         Outcome    = if_else(Survived == 1, "Survived", "Died"))
colSums(is.na(clean))                               # 每一欄還有幾個缺值

## 步驟 4　探索：誰比較容易活下來？

**🤖 給 AI 的 prompt**

> 請用 clean 和 group_by() + summarise() 算出：
> 1. 整體的人數、存活人數和存活率。
> 2. 依性別（Sex）、艙等（Pclass）各自分組的人數、存活人數與存活率。
> 3. 艙等 × 性別的存活率，用 pivot_wider() 排成艙等一列、性別一欄的表。
> 4. 依年齡組（AgeGroup）、家庭人數（FamilySize）分組的人數與存活率。
> 每一個結果分開放在不同的儲存格，存活率取到小數 3 位。

In [ ]:
clean |>
  summarise(n = n(), survived = sum(Survived),
            rate = round(mean(Survived), 3))        # 0/1 的平均 = 1 的比例

In [ ]:
clean |>
  group_by(Sex) |>
  summarise(n = n(), survived = sum(Survived), rate = round(mean(Survived), 3))

In [ ]:
clean |>
  group_by(Pclass) |>
  summarise(n = n(), survived = sum(Survived), rate = round(mean(Survived), 3))

In [ ]:
clean |>
  group_by(Pclass, Sex) |>
  summarise(rate = round(mean(Survived), 3), .groups = "drop") |>
  pivot_wider(names_from = Sex, values_from = rate)     # 長變寬：性別變成欄

In [ ]:
clean |>
  group_by(AgeGroup) |>
  summarise(n = n(), rate = round(mean(Survived), 3))

In [ ]:
clean |>
  group_by(FamilySize) |>
  summarise(n = n(), rate = round(mean(Survived), 3))

## 步驟 5　畫圖

**🤖 給 AI 的 prompt**

> 請用 ggplot2 畫三張圖，圖上文字用英文，主題用 theme_bw()：
> 1. 性別、艙等各自的存活率長條圖，加上 95% 信賴區間的誤差線，用 facet 左右並排，y 軸 0 到 1。
> 2. 艙等 × 性別的存活率長條圖（dodge），柱子上標出數值。
> 3. 存活與死亡乘客的年齡分布：依 Outcome 上色的堆疊直方圖，每 5 歲一格。
> female 用綠色、male 用橘色，兩張圖要一致。

In [ ]:
# 存活率和 95% 信賴區間：rate ± 1.96 × 標準誤
rate_ci <- function(d) {
  d |>
    summarise(n = n(), rate = mean(Survived)) |>
    mutate(se = sqrt(rate * (1 - rate) / n),
           lo = rate - 1.96 * se,
           hi = rate + 1.96 * se)
}
rates <- bind_rows(
  clean |> group_by(group = Sex) |> rate_ci() |> mutate(by = "Sex"),
  clean |> group_by(group = paste("Class", Pclass)) |> rate_ci() |> mutate(by = "Passenger class")
) |>
  mutate(by = fct_inorder(by))                                  # 面板順序：Sex 在左

ggplot(rates, aes(x = group, y = rate, fill = group)) +
  geom_col(show.legend = FALSE) +
  geom_errorbar(aes(ymin = lo, ymax = hi), width = 0.2) +       # 95% 信賴區間
  facet_wrap(~ by, scales = "free_x") +                         # 左右兩張
  scale_fill_manual(values = c(female = "#66c2a5", male = "#fc8d62",
                               "Class 1" = "#1d3557", "Class 2" = "#457b9d", "Class 3" = "#8ecae6")) +
  scale_y_continuous(limits = c(0, 1)) +
  labs(title = "Survival rate (with 95% CI)", x = NULL, y = "Survival rate") +
  theme_bw()

In [ ]:
clean |>
  group_by(Pclass, Sex) |>
  summarise(rate = mean(Survived), .groups = "drop") |>
  ggplot(aes(x = Pclass, y = rate, fill = Sex)) +
  geom_col(position = "dodge") +
  geom_text(aes(label = round(rate, 2)),
            position = position_dodge(width = 0.9), vjust = -0.4) +   # 柱子上標數值
  scale_fill_manual(values = c(female = "#66c2a5", male = "#fc8d62")) +
  scale_y_continuous(limits = c(0, 1.05)) +
  labs(title = "Survival rate by class and sex", x = "Passenger class", y = "Survival rate") +
  theme_bw()

In [ ]:
ggplot(clean, aes(x = Age, fill = Outcome)) +
  geom_histogram(binwidth = 5, boundary = 0, colour = "white") +     # 每 5 歲一格，預設就是堆疊
  scale_fill_manual(values = c(Died = "#9aa5b1", Survived = "#2a9d8f")) +
  labs(title = "Age of passengers", x = "Age (years)", y = "Passengers") +
  theme_bw()

## 步驟 6　統計檢定

**🤖 給 AI 的 prompt**

> 我想回答三個問題：
> 1. 性別和存活有沒有關係？
> 2. 艙等和存活有沒有關係？
> 3. 存活者和死亡者的年齡有沒有差別？
> 請先說明每個問題適合用哪一種統計檢定、為什麼，再寫 R 程式。

In [ ]:
tab <- table(clean$Sex, clean$Survived)     # 2 × 2 列聯表
tab
chisq.test(tab)

In [ ]:
tab2 <- table(clean$Pclass, clean$Survived)  # 3 × 2 列聯表
tab2
chisq.test(tab2)

In [ ]:
clean |>
  group_by(Outcome) |>
  summarise(n = sum(!is.na(Age)), median_age = median(Age, na.rm = TRUE))
wilcox.test(Age ~ Survived, data = clean)     # = Mann–Whitney U 檢定

## 步驟 7　多變數模型：logistic regression

**🤖 給 AI 的 prompt**

> 性別、艙等、年齡彼此有關（例如頭等艙的乘客年紀比較大），分開檢定可能會互相混淆。
> 請用 glm() 做 logistic regression：Survived ~ Sex + Pclass + Age，
> 再用 broom::tidy() 算出每個變數的 odds ratio、95% 信賴區間和 p 值，整理成一張表，並用白話解釋 odds ratio 的意思。

In [ ]:
fit <- glm(Survived ~ Sex + Pclass + Age, data = clean, family = binomial)
nobs(fit)                                             # 實際用到的乘客數（Age 缺值的列會被排除）

or_table <- broom::tidy(fit, exponentiate = TRUE, conf.int = TRUE) |>   # exponentiate：係數取 e 次方 = OR
  select(term, OR = estimate, conf.low, conf.high, p.value) |>
  mutate(across(c(OR, conf.low, conf.high), \(x) round(x, 3)))
or_table

**🤖 給 AI 的 prompt**

> 請用 ggplot2 把上面的 odds ratio 畫成森林圖：每個變數一列，點是 OR，橫線是 95% 信賴區間，
> x 軸用對數尺度，在 OR = 1 畫一條虛線，變數名稱改成好讀的英文，不要畫截距。

In [ ]:
or_table |>
  filter(term != "(Intercept)") |>                     # 截距不畫
  mutate(term = recode(term,
                       Sexmale = "Male (vs female)",
                       Pclass2 = "2nd class (vs 1st)",
                       Pclass3 = "3rd class (vs 1st)",
                       Age     = "Age (per 1 year)"),
         term = fct_rev(fct_inorder(term))) |>         # 第一個變數放最上面
  ggplot(aes(x = OR, y = term)) +
  geom_vline(xintercept = 1, linetype = "dashed", colour = "grey50") +   # OR = 1：沒有關係
  geom_errorbar(aes(xmin = conf.low, xmax = conf.high), width = 0.2) +
  geom_point(size = 3, colour = "#264653") +
  scale_x_log10() +
  labs(title = "Logistic regression", x = "Odds ratio of survival (95% CI, log scale)", y = NULL) +
  theme_bw()

## 步驟 8　結論

**🤖 給 AI 的 prompt**

> 以下是我所有的分析結果：
> （貼上：整體存活率、性別與艙等的存活率表、卡方檢定與 Wilcoxon 檢定的結果、logistic regression 的 OR 表）
> 請用繁體中文寫一段約 250 字的結論，給沒有統計背景的人看：
> 1. 只能使用我提供的數字，不要自己補數字。
> 2. 分成「主要發現」和「這份分析的限制」兩段。
> 3. 不要寫出「因為是女性所以活下來」這種因果的說法。

（把 AI 寫好、你逐句核對過數字的結論貼在這裡）